# PROGRAMAS PARA LA ESTIMACIÓN DE CS-MAPs

In [ ]:
from pprint import pprint
from itertools import combinations
from multiprocessing import Pool, cpu_count
from random import sample, seed

import numpy as np
from numpy.random import default_rng
import matplotlib.pyplot as plt
from scipy.optimize import minimize, root, least_squares, Bounds, LinearConstraint
from scipy.stats import pearsonr
from scipy.special import factorial, stirling2
from scipy.linalg import eig, inv, expm, LinAlgError

from lifelines import KaplanMeierFitter

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

RANDOM_SEED = 123456
#np.random.seed(RANDOM_SEED)

## Preprocessing of $\texttt{colorectal}$

In [ ]:
# Reads the file after being exported from R into .csv format
colorectal = pd.read_csv("colorectal.csv")

# Initializes the arrays where we will store the data, using numpy
num_patients = colorectal["id"].nunique()
times = -np.ones((num_patients, len(colorectal))) # The idea is to leave enough space for recording times
num_times = -np.ones(num_patients, dtype=int)
last_state = -np.ones(num_patients, dtype=int)

# Fills in the dataset
for id in range(1, num_patients + 1):
    # Groups by patient id, and collects number of times for each patient id
    patient_data = colorectal[colorectal['id'] == id].copy()
    num_times[id - 1] = len(patient_data)
    patient_data = patient_data.sort_values('time1')
    
    # Fills in the times with the values of the "gap.time" variable
    times[id - 1, :num_times[id - 1]] = patient_data['gap.time'].values
    
    # Fills in the last state values with a 0 if there is recurrence or censorship and 1 if there is a death
    last_state[id - 1] = patient_data['state'].iloc[-1]

# Truncates the times variable
max_num_times = num_times.max()
times = times[:, :max_num_times]

# Converts to long-form DataFrame
records = []
for i in range(num_patients):
    patient_data = {"id": i + 1}
    for j in range(0, max_num_times):
        patient_data[f"T{j+1}"] = times[i, j]
    patient_data["N"] = num_times[i]
    patient_data["last_state"] = last_state[i]
    
    records.append(patient_data)

colorectal = pd.DataFrame(records)

# Saves to CSV
colorectal.to_csv('colorectal_PREPROCESSED.csv', index=False)

## Preprocessing of $\texttt{bladder1}$

In [ ]:
# Reads the file after being exported from R into .csv format
bladder = pd.read_csv("bladder1.csv")

# Initializes the arrays where we will store the data, using numpy
num_patients = bladder["id"].nunique()
times = -np.ones((num_patients, len(bladder))) # The idea is to leave enough space for recording times
num_times = -np.ones(num_patients, dtype=int)
last_state = -np.ones(num_patients, dtype=int)

# Fills in the dataset
for id in range(1, num_patients + 1):
    # Groups by patient id, and collects number of times for each patient id
    patient_data = bladder[bladder['id'] == id].copy()
    num_times[id - 1] = len(patient_data)
    patient_data = patient_data.sort_values('stop')
    
    # Fills in the times with the values of the "gap.time" variable
    times[id - 1, :num_times[id - 1]] = patient_data["stop"].values - patient_data["start"].values
    
    # Fills in the last state values with a 0 if there is recurrence or censorship and 1 if there is a death
    last_state[id - 1] = 0 if patient_data['status'].iloc[-1] in [0, 1] else 1 

# Truncates the times variable
max_num_times = num_times.max()
times = times[:, :max_num_times]

# Converts to long-form DataFrame
records = []
for i in range(num_patients):
    patient_data = {"id": i + 1}
    for j in range(0, max_num_times):
        patient_data[f"T{j+1}"] = times[i, j]
    patient_data["N"] = num_times[i]
    patient_data["last_state"] = last_state[i]
    
    records.append(patient_data)

df_preprocessed = pd.DataFrame(records)

# Saves to CSV
df_preprocessed.to_csv('bladder1_PREPROCESSED.csv', index=False)

## Extracting empirical statistics from datasets

In [ ]:
def empirical_r_moments(dataframe, k=1, censorship=False):
    """
    Computes the sample moment of order k the of number of recurrences before death r, from a preprocessed wide-format DataFrame.
    
    Parameters:
    - dataframe: preprocessed DataFrame (with T1, T2, ..., TN, 'N', 'last_state')
    - k: order of the desired moment of r
    - censorship: If False, excludes instances of censored data from the calculations (even if data for prior times exist)
    
    Returns:
    - moment: sample moment of order k of r.
    """
    # Filter only patients who eventually died (last_state == 1)
    if censorship == False:
        dataframe = dataframe[dataframe["last_state"] == 1]

    # Calculate requested moment
    moment = np.mean((dataframe["N"] - 1) ** k)

    return moment

def empirical_conditional_times_moments(dataframe, n, l, k, equality=False, censorship=False):
    """
    Computes the sample k-th order moment of the inter-event time T_n conditioned to r>=l, i.e. bar(T_n)|r>=l, from a preprocessed wide-format DataFrame.

    Parameters:
    - dataframe: preprocessed DataFrame (with T1, T2, ..., TN, 'N', 'last_state')
    - n: index of the desired time
    - l: minimum number of recurrences after n. You must enter l>=n-1 (r=n-1 implies that T_n is a death time).
    - k: order of the desired moment
    - equality: boolean that imposes the r==n-1 condition if True, and the r>=l condition if False
    - censorship: If False, excludes instances of censored data from the calculations (even if data for prior times exist)

    Returns:
    - moment: sample k-th order moment of the inter-event time T_n conditioned to r>=l.
    """
    # Check for a combination of invalid values
    if l < n-1:
        raise KeyError("The number of recurrences before death cannot be less than the time's index minus one")
    
    # Filter only patients who eventually died (last_state == 1)
    if censorship == False:
        dataframe = dataframe[dataframe["last_state"] == 1]
        
    # Filter only patients with the desired condition
    if equality==False:
        dataframe = dataframe[(dataframe["N"] - 1) >= l]
    else:
        dataframe = dataframe[(dataframe["N"] - 1) == n - 1]

    # Calculate requested moment
    moment = np.mean(dataframe[f"T{n}"][dataframe[f"T{n}"] != -1] ** k)

    return moment

def empirical_conditional_times_medians(dataframe, n, l, equality=False, censorship=False):
    """
    Computes the sample median of the inter-event time T_n conditioned to r>=l, i.e. med(T_n)|r>=l, from a preprocessed wide-format DataFrame.

    Parameters:
    - dataframe: preprocessed DataFrame (with T1, T2, ..., TN, 'N', 'last_state')
    - n: index of the desired time
    - l: minimum number of recurrences after n. You must enter l>=n-1 (r=n-1 implies that T_n is a death time).
    - equality: boolean that imposes the r==n-1 condition if True, and the r>=l condition if False
    - censorship: If False, excludes instances censored data from the calculations (even if data for prior times exist)

    Returns:
    - median: sample median of the inter-event time T_n conditioned to r>=l.
    """
    # Check for a combination of invalid values
    if l < n-1:
        raise KeyError("The number of recurrences before death cannot be less than the time's index minus one")
    
    # Filter only patients who eventually died (last_state == 1)
    if censorship == False:
        dataframe = dataframe[dataframe["last_state"] == 1]
        
    # Filter only patients with the desired condition
    if equality==False:
        dataframe = dataframe[(dataframe["N"] - 1) >= l]
    else:
        dataframe = dataframe[(dataframe["N"] - 1) == n - 1]

    # Calculate requested median
    median = np.median(dataframe[f"T{n}"][dataframe[f"T{n}"] != -1])

    return median

def empirical_conditional_times_correlations(dataframe, n, p, l, equality=False, censorship=False):
    """
    Computes the sample Pearson correlation of inter-event times T_n, T_p conditioned to r>=l, from a preprocessed wide-format DataFrame.

    Parameters:
    - dataframe: preprocessed DataFrame (with T1, T2, ..., TN, 'N', 'last_state')
    - n: index of one of the inter-event times
    - p: index of the other inter-event time.
    - l: minimum number of recurrences after n. You must enter l>=max(p-1, n-1) (r=max(p-1, n-1) implies that T_n is a death time)
    - equality: boolean that imposes the r==max(n-1, p-1) condition if True, and the r>=l condition if False
    - censorship: If False, excludes instances censored data from the calculations (even if data for prior times exist)

    Returns:
    - correlation: sample correlation of the inter-event times T_n, T_p conditioned to r>=l.
    """
    # Check for a combination of invalid values
    if l < max(n-1, p-1):
        raise KeyError("The number of recurrences before death cannot be less than the times' indices minus one")
    
    # Filter only patients who eventually died (last_state == 1)
    if censorship == False:
        dataframe = dataframe[dataframe["last_state"] == 1]
    
    # Filter only patients with the desired condition
    if equality==False:
        dataframe = dataframe[(dataframe["N"] - 1) >= l]
    else:
        dataframe = dataframe[(dataframe["N"] - 1) == max(n - 1, p - 1)]

    # Calculate requested correlation
    condition = (dataframe[f"T{n}"] != -1) & (dataframe[f"T{p}"] != -1)
    if condition.sum() < 2:
        return np.nan
    
    correlation, _ = pearsonr(dataframe[f"T{n}"][condition], dataframe[f"T{p}"][condition])

    return correlation

In [ ]:
def empirical_conditional_death_time_mean(dataframe, t_cond, censorship=False):
    """
    Empirical mean of the *total* time–to–death  T_D  conditioned on  T1 > t.

    Parameters
    ----------
    dataframe : pandas.DataFrame
        A **wide** CS-MAP sample with columns  T1 … TK,  'N',  'last_state'.
        Missing T_i values must be -1 (same convention as  simulate_csmap()).
    t_cond : float
        Conditioning threshold  (keep only rows where  T1 > t_cond).
    censorship : bool, default False
        *False* → exclude censored trajectories  (last_state == 0).  
        *True*  → include them (they contribute their censored T_D).

    Returns
    -------
    dict with keys
        "mean"       : empirical conditional mean  \bar{T_D | T1>t}
                       (``None`` if no row fulfils the condition),
        "num_times"  : number of rows used in that average
                       (0 if none were usable).

    Notes
    -----
    * When  ``censorship=False``  only rows with  last_state==1  are used.
    * ``num_times`` is the *denominator* in the sample mean.
    """
    # ─── 1 · basic validity checks ────────────────────────────────
    if "T1" not in dataframe:
        return {"mean": None, "num_times": 0}

    # ─── 2 · row-level filters ────────────────────────────────────
    ok  = (dataframe["T1"].to_numpy() > t_cond)
    if not censorship:
        ok &= (dataframe["last_state"].to_numpy() == 1)

    if not ok.any():
        return {"mean": None, "num_times": 0}

    df_ok = dataframe.loc[ok]

    # ─── 3 · build the matrix of T_i (−1 placeholders → 0) ───────
    Tcols = [c for c in df_ok.columns if c.startswith("T")]
    Tmat  = df_ok[Tcols].to_numpy(copy=True)
    Tmat[Tmat < 0.0] = 0.0           # drop the –1 “no value” markers

    TD_vals = Tmat.sum(axis=1)       # vectorised row-sums
    mean_TD = float(TD_vals.mean())
    n_used  = int(TD_vals.size)

    return {"mean": mean_TD, "num_times": n_used}

def empirical_conditional_death_time_median(dataframe, t_cond, censorship=False):
    """
    Median of the *total* time–to–death T_D conditioned on T1 > t.

    Same structure as empirical_conditional_death_time_mean.
    """
    if "T1" not in dataframe:
        return {"median": None, "num_times": 0}

    ok = (dataframe["T1"].to_numpy() > t_cond)
    if not censorship:
        ok &= (dataframe["last_state"].to_numpy() == 1)

    if not ok.any():
        return {"median": None, "num_times": 0}

    df_ok = dataframe.loc[ok]
    Tcols = [c for c in df_ok.columns if c.startswith("T")]
    Tmat = df_ok[Tcols].to_numpy(copy=True)
    Tmat[Tmat < 0.0] = 0.0

    TD_vals = Tmat.sum(axis=1)
    return {"median": float(np.median(TD_vals)), "num_times": int(TD_vals.size)}


def empirical_conditional_death_time_percentile(dataframe, t_cond, censorship=False, q=95):
    """
    Percentile of the *total* time–to–death T_D conditioned on T1 > t.

    Parameters
    ----------
    q : float
        Desired percentile (default is 95).

    Returns dict with keys:
        "percentile" : value of the q-th percentile
        "num_times"  : number of samples used
    """
    if "T1" not in dataframe:
        return {"percentile": None, "num_times": 0}

    ok = (dataframe["T1"].to_numpy() > t_cond)
    if not censorship:
        ok &= (dataframe["last_state"].to_numpy() == 1)

    if not ok.any():
        return {"percentile": None, "num_times": 0}

    df_ok = dataframe.loc[ok]
    Tcols = [c for c in df_ok.columns if c.startswith("T")]
    Tmat = df_ok[Tcols].to_numpy(copy=True)
    Tmat[Tmat < 0.0] = 0.0

    TD_vals = Tmat.sum(axis=1)
    return {"percentile": float(np.percentile(TD_vals, q)), "num_times": int(TD_vals.size)}


In [ ]:
colorectal = pd.read_csv("colorectal_PREPROCESSED.csv", index_col=0)
max_index = max(colorectal["N"])

colorectal_r_moments = []
print("RECURRENCES-UNTIL-DEATH MOMENTS:")
for k in range(1, 11):
    moment = empirical_r_moments(colorectal, k)
    colorectal_r_moments.append(moment)
    print(f"Mean(r^{k}): ", moment)

print("\n")

colorectal_time_means = []
print("TIME CONDITIONAL MEANS:")
for n in range(1, max_index + 1):
    for l in range(n-1, max_index):
        conditional_mean = empirical_conditional_times_moments(colorectal, n, l, 1)
        colorectal_time_means.append(conditional_mean)
        print(f"Mean(T_{n} | r >= {l}): ", conditional_mean)

print("\n")

colorectal_time_medians = []
print("TIME MEDIANS:")
for n in range(1, max_index + 1):
    for l in range(n-1, max_index):
        conditional_median = empirical_conditional_times_medians(colorectal, n, l)
        colorectal_time_medians.append(conditional_median)
        print(f"Median(T_{n} | r >= {l}):", conditional_median)

print("\n")

colorectal_time_correlations = []
print("TIME CORRELATIONS:")
for n, p in combinations([integer for integer in range(1, max_index + 1)], 2):
    for l in range(max(n-1, p-1), max_index):
        conditional_time_correlation = empirical_conditional_times_correlations(colorectal, n, p, l)
        colorectal_time_correlations.append(conditional_time_correlation)
        print(f"rho(T_{n}, T_{p} | r >= {l}): ", conditional_time_correlation)

In [ ]:
bladder = pd.read_csv("bladder1_PREPROCESSED.csv", index_col=0)
max_index = max(bladder["N"])

bladder_r_moments = []
print("RECURRENCES-UNTIL-DEATH MOMENTS:")
for k in range(1, 11):
    moment = empirical_r_moments(bladder, k)
    bladder_r_moments.append(moment)
    print(f"Mean(r^{k}): ", moment)

print("\n")

bladder_time_means = []
print("TIME CONDITIONAL MEANS:")
for n in range(1, max_index + 1):
    for l in range(n-1, max_index):
        conditional_mean = empirical_conditional_times_moments(bladder, n, l, 1)
        bladder_time_means.append(conditional_mean)
        print(f"Mean(T_{n} | r >= {l}): ", conditional_mean)

print("\n")

bladder_time_medians = []
print("TIME MEDIANS:")
for n in range(1, max_index + 1):
    for l in range(n-1, max_index):
        conditional_median = empirical_conditional_times_medians(bladder, n, l)
        bladder_time_medians.append(conditional_median)
        print(f"Median(T_{n} | r >= {l}):", conditional_median)

print("\n")

bladder_time_correlations = []
print("TIME CORRELATIONS:")
for n, p in combinations([integer for integer in range(1, max_index + 1)], 2):
    for l in range(max(n-1, p-1), max_index):
        conditional_time_correlation = empirical_conditional_times_correlations(bladder, n, p, l)
        bladder_time_correlations.append(conditional_time_correlation)
        print(f"rho(T_{n}, T_{p} | r >= {l}): ", conditional_time_correlation)

## CS-MAP Theoretical Formulas

In [ ]:
def parameter_swap(parameters):
    """
    Converts between {alpha0, D0, D1} and {alpha0, rates, P0, P1}. The input dict must contain either the pair (D0,D1) or (rates,P0,P1).
    """
    if "D0" in parameters and "D1" in parameters:
        # Change (alpha0, D0, D1) into (alpha0, lambda, P0, P1)
        D0, D1 = parameters["D0"], parameters["D1"]
        rates = -np.diag(D0)
        inverse_rates = np.diag(1.0/rates)
        P0 = inverse_rates @ D0 + np.eye(D0.shape[0])
        P1 = inverse_rates @ D1
        return {"alpha0": parameters.get("alpha0"), "rates": rates, "P0": P0, "P1": P1}
    
    elif "rates" in parameters and "P0" in parameters and "P1" in parameters:
        # Change (alpha0, lambda, P0, P1) into (alpha0, D0, D1)
        rates, P0, P1 = parameters["rates"], parameters["P0"], parameters["P1"]
        D0 = np.diag(rates) @ (P0 - np.eye(P0.shape[0]))
        D1 = np.diag(rates) @ P1
        return {"alpha0": parameters.get("alpha0"), "D0": D0, "D1": D1}
    
    else:
        # Errors exit
        raise ValueError("parameter dict must contain either D0 & D1 "
                         "or rates, P0 & P1")

testing_dictionary = {"alpha0": np.array([0.5, 0.5, 0]),
                      "D0": np.array([[-1.9, 0, 0],
                                      [0.0033, -0.11, 0.1056],
                                      [0, 0, -0.13]]),
                       "D1": np.array([[1.71, 0.19, 0],
                                       [0, 0, 0.0011],
                                       [0, 0.104, 0.026]])}
parameter_swap(testing_dictionary)

In [ ]:
def theoretical_r_moments(alpha0, D0, D1, k):
    """
    Given CS-MAP parameters, computes the moment of order k of the number of recurrences r, i.e. E(r^k).
    
    Parameters:
    - alpha0: 1D array of shape (m,), initial distribution
    - D0, D1: matrices of shape (m, m), where D0 is invertible, D0's diagonal is negative, D0's off-diagonal and D1's elements are non-negative, and D0+D1 is row stochastic
    - k: order of the moment
    
    Returns:
    - moment: moment of order k of r, E(r^k)
    """
    m = len(alpha0)
    
    # Compute the base matrix, P*I_R(I-P*I_R)^(-1)
    P_star = - np.linalg.inv(D0) @ D1
    P_star_reduced = P_star.copy()
    P_star_reduced[:, -1] = 0
    resolvent = np.linalg.inv(np.eye(m) - P_star_reduced)
    M = P_star_reduced @ resolvent

    # Compute powers of the base matrix
    M_powers = [np.eye(m)]
    for j in range(1, k + 1):
        M_powers.append(M_powers[j-1] @ M)

    # Sum the powers of the helper matrix and compute the moment
    moment_matrix = np.zeros((m, m))
    for j in range(1, k + 1):
        moment_matrix += factorial(j) * stirling2(k, j, exact=True) * M_powers[j]
    moment = np.sum(alpha0 @ moment_matrix)

    return moment

In [ ]:
# --------------------------------------------------------------------
# helpers
# --------------------------------------------------------------------
def _matrix_power(A, k):
    """Fast integer power (binary-squaring, O(log k))."""
    result = np.eye(A.shape[0])
    base   = A.copy()
    exp    = k
    while exp:
        if exp & 1:
            result = result @ base
        base = base @ base
        exp >>= 1
    return result


# --------------------------------------------------------------------
# 95-th percentile of r
# --------------------------------------------------------------------
def theoretical_r_percentile(alpha0, D0, D1, q=0.5, tol=1e-12,  max_n=1_000_000):
    """
    Return the *smallest integer n* such that
        1 – α₀ (P* I_R)^{n+1} 1  ≥  q   (default q=0.95)

    Parameters
    ----------
    alpha0 : 1-D array (m,)
    D0, D1 : (m,m) CS-MAP matrices
    q      : quantile in (0,1)
    tol    : numerical tolerance for the CDF comparison
    max_n  : hard cap to avoid infinite loops if the chain is ill-posed
    """
    m = len(alpha0)

    # ----- build  P*I_R  -------------------------------------------
    P_star = -np.linalg.inv(D0) @ D1              # embedded DTMC
    PIR    = P_star.copy()
    PIR[:, -1] = 0.0                              # zero transitions to the exit state
    PIR    = PIR[:-1, :-1]                        # keep the (m-1)×(m-1) recurrence block

    # initial distribution restricted to the recurrence set
    alphaR = alpha0[:-1] / alpha0[:-1].sum()
    ones   = np.ones(m-1)

    # quick lambda to compute CDF(n) = 1 – α (PIR)^{n+1} 1
    def cdf(n: int) -> float:
        tail = (alphaR @ _matrix_power(PIR, n+1) @ ones).item()
        return 1.0 - tail

    # guard against an ill-posed system
    if cdf(max_n) + tol < q:
        raise RuntimeError("max_n too small or the chain hardly exits; "
                           "increase `max_n` or re-check the parameters.")

    # ----- exponential search for an upper bound where CDF ≥ q ----
    n_hi = 0
    while cdf(n_hi) + tol < q:
        n_hi = 2 * n_hi + 1
        if n_hi > max_n:
            n_hi = max_n
            break
    n_lo = max(0, (n_hi - 1) // 2)

    # ----- integer bisection --------------------------------------
    while n_lo + 1 < n_hi:
        mid = (n_lo + n_hi) // 2
        if cdf(mid) + tol >= q:
            n_hi = mid
        else:
            n_lo = mid

    return n_hi

alpha0 = np.array([0.5, 0.5, 0])
D0 = np.array([[-1.9, 0, 0],
               [0.0033, -0.11, 0.1056],
               [0, 0, -0.13]])
D1 = np.array([[1.71, 0.19, 0],
               [0, 0, 0.0011],
               [0, 0.104, 0.026]])

theoretical_r_percentile(alpha0, D0, D1, q=0.5)

In [ ]:
def theoretical_times_moments(alpha0, D0, D1, n, l, k=1, equality=False):
    """
    Given CS-MAP parameters, computes the k-th order moment of the n-th inter-event time T_n, conditioned to r>=l, i.e. E(T_n^k | r>=l). The condition r=n-1 may also be input.
    
    Parameters:
    - alpha0: 1D array of shape (m,), initial distribution
    - D0, D1: matrices of shape (m, m), where D0 is invertible, D0's diagonal is negative, D0's off-diagonal and D1's elements are non-negative, and D0+D1 is row stochastic
    - n: index of the inter-event time T_n
    - l: minimum number of recurrences after n. You must enter l>=n-1 (r=n-1 implies that T_n is a death time)
    - k: order of moment
    - equality: boolean that imposes the r==n-1 condition if True, and the r>=l condition if false

    Returns:
    - moment: scalar, E(T_n^k | r >= l)
    """
    # Calculate the inverse of D0
    inv_D0 = np.linalg.inv(D0)

    # Calculate the reduced transition matrix
    P_star = - inv_D0 @ D1
    P_star_recurrence = P_star.copy()
    P_star_recurrence[:, -1] = 0

    # Decide whether to calculate with the condition r>=l or r==n-1
    if equality==False:
        # Calculate the moment
        numerator = np.sum(alpha0 @ np.linalg.matrix_power(P_star_recurrence, n - 1) @ np.linalg.matrix_power(inv_D0, k) @ np.linalg.matrix_power(P_star_recurrence, l - (n - 1)))
        denominator = np.sum(alpha0 @ np.linalg.matrix_power(P_star_recurrence, l))
    else:
        # Construct the other reduced transition matrix
        m = len(alpha0)
        P_star_death = np.zeros((m, m))
        P_star_death[:, -1] = P_star[:, -1]

        # Calculate the moment
        numerator = np.sum(alpha0 @ np.linalg.matrix_power(P_star_recurrence, n - 1) @ np.linalg.matrix_power(inv_D0, k) @ P_star_death)
        denominator = np.sum(alpha0 @ np.linalg.matrix_power(P_star_recurrence, n - 1) @ P_star_death)

    moment_no_sign = factorial(k) * numerator / denominator
    moment = - moment_no_sign if k%2!=0 else moment_no_sign

    return moment

In [ ]:
def theoretical_times_correlations(alpha0, D0, D1, n, p, l, equality=False):
    """
    Given CS-MAP parameters, computes the correlation of the n-th and p-th inter-event times T_n, T_p conditioned to r>=l, i.e. rho(T_n, T_p | r>=l). The condition r=n-1 may also be input.

    Parameters:
    - alpha0: 1D array of shape (m,), initial distribution
    - D0, D1: matrices of shape (m, m), where D0 is invertible, D0's diagonal is negative, D0's off-diagonal and D1's elements are non-negative, and D0+D1 is row stochastic
    - n: first index of the inter-event times
    - p: second index of the inter-event times (this is the bigger one)
    - l: minimum number of recurrences after max(n, p). You must enter l>=max(n-1, p-1) (r=max(n-1, p-1) implies that T_max(n-1, p-1) is a death time)
    - equality: boolean that imposes the r==max(n-1, p-1) condition if True, and the r>=l condition if False.

    Returns:
    - moment: scalar, rho(T_n, T_p | r >= l)
    """
    # Calculate the inverse of D0
    inv_D0 = np.linalg.inv(D0)
    
    # Calculate the reduced transition matrix
    P_star = - inv_D0 @ D1
    P_star_recurrence = P_star.copy()
    P_star_recurrence[:, -1] = 0
    
    # Order the indices
    if n > p:
        n, p = p, n

    # Calculates the core elements of the correlation formula
    if equality==False:
        # Calculate the moments
        ETT = np.sum(alpha0 @ np.linalg.matrix_power(P_star_recurrence, n - 1) @ inv_D0 @ np.linalg.matrix_power(P_star_recurrence, p - n) @ inv_D0 @ np.linalg.matrix_power(P_star_recurrence, l - (p - 1))) / np.sum(alpha0 @ np.linalg.matrix_power(P_star_recurrence, l))
        ET1 = theoretical_times_moments(alpha0, D0, D1, n, l, 1, equality)
        ET2 = theoretical_times_moments(alpha0, D0, D1, p, l, 1, equality)
        ET1_sq = theoretical_times_moments(alpha0, D0, D1, n, l, 2, equality)
        ET2_sq = theoretical_times_moments(alpha0, D0, D1, p, l, 2, equality)
    else:
        # Construct the other reduced transition matrix
        m = len(alpha0)
        P_star_death = np.zeros((m, m))
        P_star_death[:, -1] = P_star[:, -1]

        # Calculate the moments
        ETT = np.sum(alpha0 @ np.linalg.matrix_power(P_star_recurrence, n - 1) @ inv_D0 @ np.linalg.matrix_power(P_star_recurrence, p - n) @ inv_D0 @ P_star_death) / np.sum(alpha0 @ np.linalg.matrix_power(P_star_recurrence, p - 1) @ P_star_death)
        ET1 = theoretical_times_moments(alpha0, D0, D1, n, l, 1, equality)
        ET2 = theoretical_times_moments(alpha0, D0, D1, p, l, 1, equality)
        ET1_sq = theoretical_times_moments(alpha0, D0, D1, n, l, 2, equality)
        ET2_sq = theoretical_times_moments(alpha0, D0, D1, p, l, 2, equality)

    # Compute correlation coefficient
    cov = ETT - ET1 * ET2
    std_ET1 = np.sqrt(ET1_sq - ET1 ** 2)
    std_ET2 = np.sqrt(ET2_sq - ET2 ** 2)
    correlation = cov / (std_ET1 * std_ET2)

    return correlation

In [ ]:
def theoretical_times_moments_OLD(alpha0, D0, D1, k, n, l=None):
    """
    Computes the k-th moment of the n-th inter-event time T_n under CS-MAP.
    
    Parameters:
    - alpha0: (m,) initial distribution
    - D0, D1: (m, m) CS-MAP generator matrices
    - k: order of moment
    - n: index of the inter-event time T_n (starts at 1)
    - l: if provided, compute E[T_n^k | r >= l]; 
         otherwise, compute E[T_n^k | r >= n - 1]
    
    Returns:
    - moment: scalar, E[T_n^k | condition]
    """
    m = len(alpha0)
    P_star = -np.linalg.inv(D0) @ D1
    
    # I_R is identity with last entry zero (absorbing death state)
    I_R = np.eye(m)
    I_R[-1, -1] = 0

    # Common prefix: alpha_0 * (P* I_R)^{n-1}
    A = P_star @ I_R
    alpha_n_minus_1 = alpha0 @ np.linalg.matrix_power(A, n - 1)

    # Middle term: (-D0)^(-k-1) D1
    D_term = np.linalg.matrix_power(-D0, -(k + 1)) @ D1

    if l is None:
        # Formula for E[T_n^k | r >= n - 1]
        numerator = factorial(k) * (alpha_n_minus_1 @ D_term @ np.ones(m))
        denominator = alpha_n_minus_1 @ np.ones(m)
    else:
        # Formula for E[T_n^k | r >= l]
        A_power = np.linalg.matrix_power(A, l - n)
        numerator = factorial(k) * (alpha_n_minus_1 @ D_term @ I_R @ A_power @ np.ones(m))
        denominator = (alpha0 @ np.linalg.matrix_power(A, l)) @ np.ones(m)
        
    return (numerator / denominator).item()

def th_cor_r_OLD(params, event_pairs=[(1, 2), (1, 3), (2, 3)], last_recurrence=False):
    """
    Computes theoretical correlations between times to events in a Markov or phase-type process.

    Args:
        params: dict containing 'alpha0' (initial distribution), 'D0' (base transition matrix),
                and 'D1' (event-triggering transition matrix).
        event_pairs: list of tuples representing (event_i, event_j) pairs to compute correlations for.
        last_recurrence: if True, uses recurrence-based normalization in the computation.

    Returns:
        np.ndarray of correlation values for each event pair.
    """
    
    # Initial probability vector
    alpha0 = np.array(params['alpha0']).reshape(1, -1)
    
    # Matrix inversions and core transition matrix setup
    inv_D0 = np.linalg.inv(params['D0'])
    P_star = -inv_D0 @ params['D1']  # Fundamental transition probabilities
    
    # Remove transitions to the absorbing state (last column)
    P_star_reduced = P_star.copy()
    P_star_reduced[:, -1] = 0

    # Matrix of expected first and second moments
    M2 = -inv_D0 @ P_star
    M3 = -inv_D0 @ M2

    M2_reduced = M2.copy()
    M2_reduced[:, -1] = 0
    M3_reduced = M3.copy()
    M3_reduced[:, -1] = 0

    correlations = np.zeros(len(event_pairs))

    for i, (start_event, end_event) in enumerate(event_pairs):
        # Evolve the process to just before the first event
        prob_vec = alpha0.copy()
        for _ in range(start_event - 1):
            prob_vec = prob_vec @ P_star_reduced

        # First and second moment vectors at the start
        first_moment_vec = prob_vec @ M2_reduced
        second_moment_vec = prob_vec @ M3_reduced

        # Propagate these vectors up to just before the second event
        num_steps = end_event - start_event - 1
        for _ in range(num_steps):
            first_moment_vec = first_moment_vec @ P_star_reduced
            second_moment_vec = second_moment_vec @ P_star_reduced

        # Final propagation of probability vector up to second event
        for _ in range(end_event - start_event):
            prob_vec = prob_vec @ P_star_reduced

        if last_recurrence:
            denom = np.sum(prob_vec @ P_star_reduced)
            ET1 = np.sum(first_moment_vec @ P_star_reduced) / denom
            ET1_sq = 2 * np.sum(second_moment_vec @ P_star_reduced) / denom
            ET2 = np.sum(prob_vec @ M2_reduced) / denom
            ET2_sq = 2 * np.sum(prob_vec @ M3_reduced) / denom
            ETT = np.sum(first_moment_vec @ M2_reduced) / denom
        else:
            denom = np.sum(prob_vec)
            ET1 = np.sum(first_moment_vec) / denom
            ET1_sq = 2 * np.sum(second_moment_vec) / denom
            ET2 = np.sum(prob_vec @ M2) / denom
            ET2_sq = 2 * np.sum(prob_vec @ M3) / denom
            ETT = np.sum(first_moment_vec @ M2) / denom

        # Compute correlation coefficient
        cov = ETT - ET1 * ET2
        std_ET1 = np.sqrt(ET1_sq - ET1 ** 2)
        std_ET2 = np.sqrt(ET2_sq - ET2 ** 2)
        correlations[i] = cov / (std_ET1 * std_ET2)

    return correlations

In [ ]:
alpha0 = np.array([0.5, 0.5])
D0 = np.array([[-2.5, 0], 
               [0, -10]])
D1 = np.array([[0, 2.5], 
               [10, 0]])

alpha0 = np.array([0.5, 0.5])
D0 = np.array([[-1.5, 1.0],
               [0.5, -2.0]])
D1 = np.array([[0.5, 0.0],
               [0.0, 1.5]])

alpha0 = np.array([0.7, 0.3])
D0 = np.array([[-1.5, 1.0],
               [0.5, -2.0]])
D1 = np.array([[0.5, 0.0],
               [0.0, 1.5]])

# ESTE EJEMPLO ES DEL PAPER DE ÁLVARO.
alpha0 = np.array([1, 0, 0, 0])
D0 = np.array([[-0.2, 0, 0, 0.162],
               [0, -0.09, 0, 0],
               [0, 0, -0.73, 0],
               [0, 0, 0, -0.13]])
D1 = np.array([[0, 0.029, 0.03, 0.006],
               [0.03, 0.06, 0, 0],
               [0, 0, 0.64, 0.09],
               [0, 0, 0.022, 0.108]])

# ESTE EJEMPLO ES DEL TFM DE ÁLVARO
alpha0 = np.array([0.5, 0.5, 0])
D0 = np.array([[-1.9, 0, 0],
               [0.0033, -0.11, 0.1056],
               [0, 0, -0.13]])
D1 = np.array([[1.71, 0.19, 0],
               [0, 0, 0.0011],
               [0, 0.104, 0.026]])

# ESTE EJEMPLO ES TAL QUE LAS 3 PRIMERAS CORRELACIONES (r>=2) SON TODAS 0
'''alpha0 = np.array([1, 0])
D0 = np.array([[-1, 0],
               [0, -2]])
D1 = np.array([[0.5, 0.5],
               [0.5, 1.5]])'''

print(theoretical_times_moments(alpha0, D0, D1, 1, 1, 1))
#print(theoretical_times_moments_OLD(alpha0, D0, D1, 1, 1, 1))
#print(th_cor_r_OLD({"alpha0": alpha0, "D0": D0, "D1": D1}, [(1,2), (1,3), (2,3)], False))
print(theoretical_times_correlations(alpha0, D0, D1, 1, 2, 1))
print(theoretical_times_correlations(alpha0, D0, D1, 1, 2, 2))
print(theoretical_times_correlations(alpha0, D0, D1, 1, 3, 2))
print(theoretical_times_correlations(alpha0, D0, D1, 1, 3, 3))
print(theoretical_times_correlations(alpha0, D0, D1, 2, 3, 2))
print(theoretical_times_correlations(alpha0, D0, D1, 2, 3, 3))

## Simulating CS-MAPs

In [ ]:
def simulate_csmap(params, kmax, N_sim=1000, seed=RANDOM_SEED, progress=True):
    """
    Simulates inter-event times from a CS-MAP up to a maximum number of events (kmax), and returns results in wide-form pandas DataFrame.

    Parameters:
    - params: dictionary with CS-MAP parameters (alpha0, D0, D1)
    - kmax: maximum number of events (inter-event times)
    - N_sim: number of CS-MAP runs
    - seed: random seed for reproducibility
    - progress: if True, shows the current iteration on execution

    Returns:
    - A dictionary with:
        - 'id': array of simulation indices
        - 'Tn': (N_sim, kmax) array of inter-event times
        - 'N': array of number of observed events before termination
        - 'last_state': array indicating terminal state (0 = censoring, 1 = death)
    """
    # Fix the random seed
    if seed is not None:
        np.random.seed(seed)

    # Extract the CS-MAP parameters
    alpha0 = params["alpha0"]
    D0 = params["D0"]
    D1 = params["D1"]
    m = len(alpha0)

    # Normalize transition matrices
    lambda_rates = -np.diag(D0)
    P0 = D0 / lambda_rates[:, None]
    np.fill_diagonal(P0, 0)
    P1 = D1 / lambda_rates[:, None]

    # Initialize storage
    Tn = -np.ones((N_sim, kmax))
    N = -np.ones(N_sim, dtype=int)
    last_state = np.zeros(N_sim, dtype=int)

    # Run all of the simulations
    for i in range(N_sim):
        state = np.random.choice(m, p=alpha0)
        for j in range(kmax):
            t = 0.0
            while state < m:
                t += np.random.exponential(1 / lambda_rates[state]) # NumPy.random.exponential USES SCALE == 1/RATE AS ARGUMENT
                probs = np.concatenate([P0[state], P1[state]])
                probs = probs / probs.sum() # TO ACCOUNT FOR NUMERICAL ERRORS!!!!
                state = np.random.choice(2 * m, p=probs)
            Tn[i, j] = t
            # Check if the death state is visited
            if state == 2 * m - 1:
                N[i] = j + 1
                last_state[i] = 1
                break
            # If not death, a recurrence state has been visited
            else:
                state -= m
            # Assigns a censorship if kmax events have occurred (note that last_state[i]==0 already)
            if j == kmax:
                N[i] = kmax
            
        # Display progress
        if progress == True and (i+1)%(N_sim//10) == 0:
            print(f"{100*(i+1)//N_sim}% of the simulations finished!")

    # Construct the DataFrame
    dataframe = pd.DataFrame(Tn[:, :max(N)], columns=[f"T{i+1}" for i in range(max(N))])
    dataframe["N"] = N
    dataframe["last_state"] = last_state

    return dataframe

In [ ]:
alpha0 = np.array([0.5, 0.5])
D0 = np.array([[-2.5, 0], 
               [0, -10]])
D1 = np.array([[0, 2.5], 
               [10, 0]])

alpha0 = np.array([0.5, 0.5])
D0 = np.array([[-1.5, 1.0],
               [0.5, -2.0]])
D1 = np.array([[0.5, 0.0],
               [0.0, 1.5]])

alpha0 = np.array([0.7, 0.3])
D0 = np.array([[-1.5, 1.0],
               [0.5, -2.0]])
D1 = np.array([[0.5, 0.0],
               [0.0, 1.5]])

# ESTE EJEMPLO ES DEL PAPER DE ÁLVARO.
alpha0 = np.array([1, 0, 0, 0])
D0 = np.array([[-0.2, 0, 0, 0.162],
               [0, -0.09, 0, 0],
               [0, 0, -0.73, 0],
               [0, 0, 0, -0.13]])
D1 = np.array([[0, 0.029, 0.03, 0.006],
               [0.03, 0.06, 0, 0],
               [0, 0, 0.64, 0.09],
               [0, 0, 0.022, 0.108]])

# ESTE EJEMPLO ES DEL TFM DE ÁLVARO
alpha0 = np.array([0.5, 0.5, 0])
D0 = np.array([[-1.9, 0, 0],
               [0.0033, -0.11, 0.1056],
               [0, 0, -0.13]])
D1 = np.array([[1.71, 0.19, 0],
               [0, 0, 0.0011],
               [0, 0.104, 0.026]])

# ESTE EJEMPLO ES TAL QUE LAS 3 PRIMERAS CORRELACIONES (r>=2) SON TODAS 0
'''alpha0 = np.array([1, 0])
D0 = np.array([[-1, 0],
               [0, -2]])
D1 = np.array([[0.5, 0.5],
               [0.5, 1.5]])'''

results_sample = simulate_csmap(params={"alpha0": alpha0, "D0": D0, "D1": D1}, kmax=8, N_sim=1000)
#results_sample.to_csv('PRUEBAS.csv', index=False)
#display(results_sample.head(20))

# CS-MAP Reverse Engineering

In [ ]:
def draw_CSMAP_parameters(order, gamma_shape=2, gamma_scale=0.5, random_seed=RANDOM_SEED):
    """
    Returns a dictionary {alpha0, D0, D1}  for an order x order CS-MAP sampled as follows:

      alpha0 ~ Dirichlet(1, ..., 1) of length order
      lambda_i ~ Gamma(gamma_shape, gamma_scale), for each i, independently
      for each row i:
         y ~ Dirichlet(1, ... ,1)   of length 2m-1
         split  y ->  y0_off (length order-1)  |  y1 (length order)
         place y0_off into the off-diagonal element of P0[i], set P0[i,i]=0
         P1[i,:] = y1
         renormalise so that P0[i, :]+ P1[i, :] == 1
    """
    # Set the random number generator, for reproducibility
    if random_seed is not None:
        rng = default_rng(random_seed)
    else:
        rng = default_rng() # fallback to non-deterministic

    # Sample the parameters
    alpha0 = rng.dirichlet(np.ones(order))
    rates = rng.gamma(gamma_shape, gamma_scale, size=order)
    P0 = np.zeros((order, order))
    P1 = np.zeros((order, order))
    for i in range(order):
        y  = rng.dirichlet(np.ones(2 * order - 1))
        y0 = y[:order-1]           # off-diagonal probabilities for P0
        y1 = y[order-1:]           # full row for P1

        P0[i, :i] = y0[:i]
        P0[i, i+1:] = y0[i:]
        P1[i, :] = y1

        row_sum = P0[i].sum() + P1[i].sum()
        P0[i] /= row_sum
        P1[i] /= row_sum
        # P0[i,i] stays 0 by construction

    parameters = parameter_swap({"alpha0": alpha0, "rates": rates, "P0": P0, "P1": P1})
    return parameters

test_dictionary = draw_CSMAP_parameters(order=3)
for i in range(3):
    print(np.sum(test_dictionary["D0"][i]+test_dictionary["D1"][i]))

In [ ]:
# FUNCTION FOR SAMPLING E(r) VALUES

# Order of the CS-MAP
ORDER = 3

# Parameters of the rates' distribution
GAMMA_SHAPE = 2
GAMMA_SCALE = 0.5

# Lower bounds for classifying correlations as "high"
CORR_THRESHOLDS = [0.1, 0.1, 0.1]

# Tolerance level for the 0-correlations
FIRST_CORR_TOL = 1e-3
SECOND_CORR_TOL = 5*1e-3

# Tolerance level for the mean equations
MEAN_TOL = 0.95

# Maximum number of CS-MAP draws
SAMPLE_SIZE = 5000
MAX_ITERS = 6*10**8

# Controls all of the random seeds, for reproducibility
MASTER_SEED = 42

# STEP 0: Initialize RNG
seed(MASTER_SEED)

# STEP 1: Sample CS-MAPS until 0-correlation parameters are obtained
while True:
    # Sample random seed and parameters
    random_seed = sample(range(0, 6*10**8+1), 1)[0]
    parameters = draw_CSMAP_parameters(ORDER, gamma_shape=GAMMA_SHAPE, gamma_scale=GAMMA_SCALE, random_seed=random_seed)
    alpha0, D0, D1 = parameters["alpha0"], parameters["D0"], parameters["D1"]
    # Finish when we have parameters with good enough correlations
    if max(abs(theoretical_times_correlations(alpha0, D0, D1, 1, 2, 1)), abs(theoretical_times_correlations(alpha0, D0, D1, 1, 3, 2)), abs(theoretical_times_correlations(alpha0, D0, D1, 2, 3, 2))) < FIRST_CORR_TOL:
        break
print("Found 0-correlation parameters")

# STEP 2: Calculate their associated time means
mu1 = theoretical_times_moments(alpha0, D0, D1, 1, 0)
mu2 = theoretical_times_moments(alpha0, D0, D1, 2, 1)
mu3 = theoretical_times_moments(alpha0, D0, D1, 3, 2)

# STEPS 3 & 4: Sample CS-MAPS with those means and either 0-correlations or "high" correlations

# Initialize results containers
low_correlations, high_correlations = [], []
low_samples, high_samples = 0, 0
k = 0.1

for _ in range(MAX_ITERS):

    # Escape loop when SAMPLE_SIZE samples have been collected for both categories
    if min(low_samples, high_samples) >= SAMPLE_SIZE:
        break
    
    if min(low_samples, high_samples) >= k*SAMPLE_SIZE:
        k += 0.1
        print(f"Over {100*k}% of the samples collected.")
    # Sample random seed and parameters
    random_seed = sample(range(0, 6*10**8+1), 1)[0]
    parameters = draw_CSMAP_parameters(ORDER, gamma_shape=GAMMA_SHAPE, gamma_scale=GAMMA_SCALE, random_seed=random_seed)
    alpha0, D0, D1 = parameters["alpha0"], parameters["D0"], parameters["D1"]

    # Calculate theoretical means, and check whether they match the originals (i.e. they are "fixed")
    # Mean times matching is done by means of relative error because the orders of magnitude are very different
    moment1, moment2, moment3 = theoretical_times_moments(alpha0, D0, D1, 1, 0), theoretical_times_moments(alpha0, D0, D1, 2, 1), theoretical_times_moments(alpha0, D0, D1, 3, 2)
    if 1 - max(min(moment1/mu1, mu1/moment1), min(moment2/mu2, mu2/moment2), min(moment3/mu3, mu3/moment3)) < MEAN_TOL:
        continue

    # If parameters with our wanted means are obtained, calculate theoretical correlations and check whether they are high, low or neither
    rho12, rho13, rho23 = theoretical_times_correlations(alpha0, D0, D1, 1, 2, 1), theoretical_times_correlations(alpha0, D0, D1, 1, 3, 2), theoretical_times_correlations(alpha0, D0, D1, 2, 3, 2)

    # High correlations
    if (high_samples <= SAMPLE_SIZE) and (abs(rho12)>=CORR_THRESHOLDS[0] or abs(rho13)>=CORR_THRESHOLDS[1] or abs(rho23)>=CORR_THRESHOLDS[2]):
        # Collect results and display progress
        print("FOUND HIGH CORRELATIONS PARAMETERS AT ITERATION ", _)
        print("Correlations: ", rho12, rho13, rho23)
        print("Relative deviations from the means: ", min(moment1/mu1, mu1/moment1), min(moment2/mu2, mu2/moment2), min(moment3/mu3, mu3/moment3))
        print("Parameters: ", alpha0, D0, D1)
        parameters["E(r)"] = theoretical_r_moments(alpha0, D0, D1, 1)
        parameters["sqrt(V(r))"] = np.sqrt(theoretical_r_moments(alpha0, D0, D1, 2) - theoretical_r_moments(alpha0, D0, D1, 1) ** 2)
        parameters["P95(r)"] = theoretical_r_percentile(alpha0, D0, D1, q=0.95)
        parameters["Correlations"] = [rho12, rho13, rho23]
        high_correlations.append(parameters)
        high_samples += 1
    
    # Low correlations
    elif (low_samples <= SAMPLE_SIZE) and max(abs(rho12), abs(rho13), abs(rho23)) <= SECOND_CORR_TOL:
        # Collect results and display progress
        print("FOUND LOW CORRELATIONS PARAMETERS AT ITERATION ", _)
        print("Correlations: ", rho12, rho13, rho23)
        print("Relative deviations from the means: ", min(moment1/mu1, mu1/moment1), min(moment2/mu2, mu2/moment2), min(moment3/mu3, mu3/moment3))
        print("Parameters: ", alpha0, D0, D1)
        parameters["E(r)"] = theoretical_r_moments(alpha0, D0, D1, 1)
        parameters["sqrt(V(r))"] = np.sqrt(theoretical_r_moments(alpha0, D0, D1, 2) - theoretical_r_moments(alpha0, D0, D1, 1) ** 2)
        parameters["P95(r)"] = theoretical_r_percentile(alpha0, D0, D1, q=0.95)
        parameters["Correlations"] = [rho12, rho13, rho23]
        low_correlations.append(parameters)
        low_samples += 1

In [ ]:
print(mu1)
print(mu2)
print(mu3)

In [ ]:
import json
from pathlib import Path
from datetime import datetime

# ------------------------------------------------------------------
# helper: recursively convert every numpy array → Python list
# ------------------------------------------------------------------
def _np_to_list(obj):
    if isinstance(obj, (list, tuple)):
        return [_np_to_list(x) for x in obj]
    if isinstance(obj, dict):
        return {k: _np_to_list(v) for k, v in obj.items()}
    # numpy scalars / arrays
    try:
        import numpy as np
        if isinstance(obj, (np.ndarray, np.generic)):
            return obj.tolist()
    except ModuleNotFoundError:
        pass                           # numpy not imported yet
    return obj                         # anything else stays as-is

# ------------------------------------------------------------------
# 1. convert →  2. dump
# ------------------------------------------------------------------
results = {
    "timestamp": datetime.utcnow().isoformat(timespec="seconds") + "Z",
    "low_correlations":  _np_to_list(low_correlations),
    "high_correlations": _np_to_list(high_correlations),
}

out_file = Path("csmaps_results.json")
with out_file.open("w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

print(f"Saved  {len(low_correlations)} low-corr  +  "
      f"{len(high_correlations)} high-corr models →  {out_file.resolve()}")


In [ ]:
import json, numpy as np
from pathlib import Path

with Path("csmaps_results_Er.json").open(encoding="utf-8") as f:
    data              = json.load(f)
    low_correlations  = data["low_correlations"]
    high_correlations = data["high_correlations"]

# if you want numpy arrays back:
for d in low_correlations + high_correlations:
    for k in ("alpha0", "D0", "D1"):               # keys that were arrays
        d[k] = np.array(d[k])

In [ ]:
# ------------------------------------------------------------
BIN_WIDTH        = 1.0      # fixed bin width       [k,k+1)
OUTLIER_FRACTION = 1e-8     # “rare”  ⇔  < tallest/20 observations
# ------------------------------------------------------------

def _pretty_matrix(M):
    with np.printoptions(precision=3, suppress=True):
        return ("\n" + np.array2string(M, separator=", ")).replace("\n", "\n        ")

def _analyse(vals, dict_list, key_name, title):
    """
    Build the histogram, classify bins, print rare-bin observations
    (sorted by descending value) and return the values that remain
    after outlier removal, ready to be plotted.
    """
    if not vals:                           # nothing to plot
        print(f"[{title}]  no observations.")
        return np.array([])

    vals = np.asarray(vals)
    # unit-width bin edges that cover the data
    lo, hi = np.floor(vals.min()), np.ceil(vals.max())
    edges  = np.arange(lo, hi + BIN_WIDTH + 1e-9, BIN_WIDTH)
    hist, _ = np.histogram(vals, bins=edges)

    tallest  = hist.max()
    rare_bins = np.where(hist < tallest * OUTLIER_FRACTION)[0]

    if rare_bins.size:                     # report every entry in a rare bin
        print(f"\n[{title}]  observations in sparse bins "
              f"(< {tallest/OUTLIER_FRACTION:.0f} items per bin):")

        bin_id   = np.searchsorted(edges, vals, side="right") - 1
        out_mask = np.isin(bin_id, rare_bins)
        out_idx  = np.where(out_mask)[0]

        # sort indices by *descending* value of the statistic
        out_idx  = out_idx[np.argsort(vals[out_idx])[::-1]]

        for i in out_idx:
            prm = dict_list[i]
            print(f"  • {key_name} = {vals[i]:.5g}")
            print(f"    alpha0 = {prm['alpha0']}")
            print(f"    D0 = {_pretty_matrix(prm['D0'])}")
            print(f"    D1 = {_pretty_matrix(prm['D1'])}")
            if "Correlations" in prm:
                print(f"    corr = {np.array(prm['Correlations'])}\n")

    return vals

def _stats_text(data):
    """Return a 3-line string with μ, median, c_v, skew."""
    μ  = data.mean()
    med = np.median(data)
    σ  = data.std(ddof=0)
    cv = σ/μ if μ else np.nan
    g  = 3*(μ-med)/σ if σ else np.nan
    return (f"mean                 = {μ:.3g}\n"
            f"median               = {med:.3g}\n"
            f"standard deviation   = {σ:.3g}\n"
            f"variation coeff      = {cv:.3g}\n"
            f"skewness             = {g:.3g}")

def _plot_unit_hist(ax, data, colour, label):
    ax.hist(
        data,
        bins=np.arange(np.floor(data.min()),
                       np.ceil(data.max()) + BIN_WIDTH + 1e-9,
                       BIN_WIDTH),
        edgecolor="k",
        color=colour,
        alpha=0.7,
        density=True,            # ← show *density* instead of frequency
    )
    ax.set_xlabel(label)
    ax.set_ylabel("density")     # ← y-axis label matches the new scale

    # ---- add statistics box ----
    ax.text(0.98, 0.98, _stats_text(data),
            transform=ax.transAxes, va="top", ha="right",
            bbox=dict(boxstyle="round,pad=0.3", fc="w", ec=colour, lw=1))


def plot_ErVrP95(low_dicts, high_dicts):
    """
    Four 1-unit-bin histograms:
        low-corr   – E(r)            high-corr – E(r)
        low-corr   – sqrt(V(r))      high-corr – sqrt(V(r))
        low-corr   - P95(r)          high-corr - P95(r)
    Sparse-bin observations are printed and removed from the plots.
    """

    # ----------- LOW-corr group -----------
    low_E  = _analyse([d["E(r)"] for d in low_dicts],
                      low_dicts, "E(r)", "LOW E(r)")
    low_V  = _analyse([d["sqrt(V(r))"] for d in low_dicts],
                      low_dicts, "sqrt(V(r))", "LOW sqrt(V(r))")
    low_P95 = _analyse([d["P95(r)"] for d in low_dicts],
                       low_dicts, "P95(r)", "LOW P95(r)")

    # ----------- HIGH-corr group ----------
    high_E = _analyse([d["E(r)"] for d in high_dicts],
                      high_dicts, "E(r)", "HIGH E(r)")
    high_V = _analyse([d["sqrt(V(r))"] for d in high_dicts],
                      high_dicts, "sqrt(V(r))", "HIGH sqrt(V(r))")
    high_P95 = _analyse([d["P95(r)"] for d in high_dicts],
                       high_dicts, "P95(r)", "HIGH P95(r)")

    # ---------- plotting ----------
    if low_E.size:
        fig, ax = plt.subplots(figsize=(6,3))
        _plot_unit_hist(ax, low_E, "gray", r"$E(r)$")
        ax.set_title(f"Low correlations – $E(r)$  (n = {len(low_E)})")
        plt.tight_layout();  plt.show()

    if low_V.size:
        fig, ax = plt.subplots(figsize=(6,3))
        _plot_unit_hist(ax, low_V, "gray", r"$sqrt(V(r))$")
        ax.set_title(f"Low correlations – $sqrt(V(r))$  (n = {len(low_V)})")
        plt.tight_layout();  plt.show()
    
    if low_P95.size:
        fig, ax = plt.subplots(figsize=(6,3))
        _plot_unit_hist(ax, low_P95, "gray", r"$P95(r)$")
        ax.set_title(f"Low correlations – $P95(r)$  (n = {len(low_P95)})")
        plt.tight_layout();  plt.show()

    if high_E.size:
        fig, ax = plt.subplots(figsize=(6,3))
        _plot_unit_hist(ax, high_E, "steelblue", r"$E(r)$")
        ax.set_title(f"High correlations – $E(r)$  (n = {len(high_E)})")
        plt.tight_layout();  plt.show()

    if high_V.size:
        fig, ax = plt.subplots(figsize=(6,3))
        _plot_unit_hist(ax, high_V, "steelblue", r"$sqrt(V(r))$")
        ax.set_title(f"High correlations – $sqrt(V(r))$  (n = {len(high_V)})")
        plt.tight_layout();  plt.show()
    
    if high_P95.size:
        fig, ax = plt.subplots(figsize=(6,3))
        _plot_unit_hist(ax, high_P95, "steelblue", r"$P95(r)$")
        ax.set_title(f"High correlations – $P95(r)$  (n = {len(high_P95)})")
        plt.tight_layout();  plt.show()

plot_ErVrP95(low_correlations, high_correlations)

In [ ]:
# FUNCTION FOR SAMPLING T_D VALUES

# Order of the CS-MAPs
ORDER = 3

# Parameters of the rates' distribution
GAMMA_SHAPE = 4.5
GAMMA_SCALE = 5

# Lower bounds for classifying correlations as "high"
CORR_THRESHOLDS = [0.1, 0.1, 0.1]

# Tolerance level for the 0-correlations
CORR_TOL = 1e-4

# Tolerance level for the mean equation
MEAN_TOL = 0.15

# Maximum number of CS-MAP draws
SAMPLE_SIZE = 5000
MAX_ITERS = 9*10**8

# Maximum number of recurrences per CS-MAP iteration
KMAX = 200

# Number of CS-MAP runs for averaging. This parameter is crucial
N_PATHS = 10**5

# Lower bound on T1, the t in E(T_D | T_1 > t)
COND_TIME = 0.5

# Fixed value of E(r)
FIXED_MEAN = 2.15

# Controls all of the random seeds, for reproducibility
MASTER_SEED = 42

# STEP 0: Initialize RNG
seed(MASTER_SEED)                                   # Python RNG
np.random.seed(MASTER_SEED)                         # NumPy RNG

# STEPS 1 & 2: Sample CS-MAPS with those means and either 0-correlations or "high" correlations

# Initialize results containers
low_correlations, high_correlations = [], []
low_samples, high_samples = 0, 0
for _ in range(MAX_ITERS):

    # Escape loop when SAMPLE_SIZE samples have been collected for both categories
    if min(low_samples, high_samples) >= SAMPLE_SIZE:
        break
    
    # Sample random seed and parameters
    random_seed = sample(range(0, 6*10**8+1), 1)[0]
    parameters = draw_CSMAP_parameters(ORDER, gamma_shape=GAMMA_SHAPE, gamma_scale=GAMMA_SCALE, random_seed=random_seed)
    alpha0, D0, D1 = parameters["alpha0"], parameters["D0"], parameters["D1"]

    # Calculate theoretical mean, and check whether it matches the original (i.e. it is "fixed")
    # Mean times matching is done by means of relative error because the orders of magnitude are very different
    mean = theoretical_r_moments(alpha0, D0, D1, 1)
    if abs(FIXED_MEAN - mean) > MEAN_TOL:
        continue

    # If parameters with our wanted means are obtained, calculate theoretical correlations and check whether they are high, low or neither
    rho12, rho13, rho23 = theoretical_times_correlations(alpha0, D0, D1, 1, 2, 1), theoretical_times_correlations(alpha0, D0, D1, 1, 3, 2), theoretical_times_correlations(alpha0, D0, D1, 2, 3, 2)

    # High correlations
    if (high_samples < SAMPLE_SIZE) and (abs(rho12)>=CORR_THRESHOLDS[0] or abs(rho13)>=CORR_THRESHOLDS[1] or abs(rho23)>=CORR_THRESHOLDS[2]):
        # Calculate E(T_D | T_1 > t) via MonteCarlo simulations
        print(f"Starting MonteCarlo simulation for high correlations (iteration {_})...")
        simulation_dataframe = simulate_csmap({"alpha0": alpha0, "D0": D0, "D1": D1}, kmax=KMAX, N_sim=N_PATHS, seed=None, progress=False)
        ETD_cond = empirical_conditional_death_time_mean(simulation_dataframe, t_cond=COND_TIME)
        medianTD_cond = empirical_conditional_death_time_median(simulation_dataframe, t_cond=COND_TIME)
        percentile95_cond = empirical_conditional_death_time_percentile(simulation_dataframe, t_cond=COND_TIME, q=95)
        if ETD_cond["mean"] is None:
            print("SIMULATION FAILED.\n")
            continue  

        # Collect results
        parameters["mean"] = ETD_cond["mean"]
        parameters["median"] = medianTD_cond["median"]
        parameters["percentile95"] = percentile95_cond["percentile"]
        parameters["Sample size"] = ETD_cond["num_times"]
        parameters["Correlations"] = [rho12, rho13, rho23]
        high_samples += 1
        high_correlations.append(parameters)

        # Display progress
        print(f"FOUND HIGH CORRELATIONS PARAMETERS AT ITERATION {_}, only {SAMPLE_SIZE-high_samples} to go!")
        print("Correlations: ", rho12, rho13, rho23)
        print("Conditional expected time until death: ", ETD_cond["mean"])
        print("Conditional median time until death: ", medianTD_cond["median"])
        print("Conditional 95th percentile time until death: ", percentile95_cond["percentile"])
        print("Sample size for the conditional expected time until death: ", ETD_cond["num_times"])
        print("Absolute deviation from the mean: ", abs(FIXED_MEAN - mean))
        print("Parameters: ", alpha0, D0, D1)
        print("\n")
    
    # Low correlations
    elif (low_samples < SAMPLE_SIZE) and max(abs(rho12), abs(rho13), abs(rho23)) <= CORR_TOL:
        # Calculate E(T_D | T_1 > t) via MonteCarlo simulations
        print(f"Starting MonteCarlo simulation for low correlations (iteration {_})...")
        simulation_dataframe = simulate_csmap({"alpha0": alpha0, "D0": D0, "D1": D1}, kmax=KMAX, N_sim=N_PATHS, seed=None, progress=False)
        ETD_cond = empirical_conditional_death_time_mean(simulation_dataframe, t_cond=COND_TIME)
        medianTD_cond = empirical_conditional_death_time_median(simulation_dataframe, t_cond=COND_TIME)
        percentile95_cond = empirical_conditional_death_time_percentile(simulation_dataframe, t_cond=COND_TIME, q=95)
        if ETD_cond["mean"] is None:
            print("SIMULATION FAILED.\n")
            continue  

        # Collect results
        parameters["mean"] = ETD_cond["mean"]
        parameters["median"] = medianTD_cond["median"]
        parameters["percentile95"] = percentile95_cond["percentile"]
        parameters["Sample size"] = ETD_cond["num_times"]
        parameters["Correlations"] = [rho12, rho13, rho23]
        low_samples += 1
        low_correlations.append(parameters)

        # Display progress
        print(f"FOUND LOW CORRELATIONS PARAMETERS AT ITERATION {_}, only {SAMPLE_SIZE-low_samples} to go!")
        print("Correlations: ", rho12, rho13, rho23)
        print("Conditional expected time until death: ", ETD_cond["mean"])
        print("Conditional median time until death: ", medianTD_cond["median"])
        print("Conditional 95th percentile time until death: ", percentile95_cond["percentile"])
        print("Sample size for the conditional expected time until death: ", ETD_cond["num_times"])
        print("Absolute deviation from the mean: ", abs(FIXED_MEAN - mean))
        print("Parameters: ", alpha0, D0, D1)
        print("\n")

# SAVE RESULTS!!!

import json
from pathlib import Path
from datetime import datetime

# ------------------------------------------------------------------
# helper: recursively convert every numpy array → Python list
# ------------------------------------------------------------------
def _np_to_list(obj):
    if isinstance(obj, (list, tuple)):
        return [_np_to_list(x) for x in obj]
    if isinstance(obj, dict):
        return {k: _np_to_list(v) for k, v in obj.items()}
    # numpy scalars / arrays
    try:
        import numpy as np
        if isinstance(obj, (np.ndarray, np.generic)):
            return obj.tolist()
        if isinstance(obj, np.generic): 
            return obj.item()
    except ModuleNotFoundError:
        pass                           # numpy not imported yet
    return obj                         # anything else stays as-is

# ------------------------------------------------------------------
# 1. convert →  2. dump
# ------------------------------------------------------------------
results = {
    "timestamp": datetime.utcnow().isoformat(timespec="seconds") + "Z",
    "low_correlations":  _np_to_list(low_correlations),
    "high_correlations": _np_to_list(high_correlations),
}

out_file = Path("csmaps_results_TD.json")
with out_file.open("w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

print(f"Saved  {len(low_correlations)} low-corr  +  "
      f"{len(high_correlations)} high-corr models →  {out_file.resolve()}")

In [ ]:
import json, numpy as np
from pathlib import Path

with Path("csmaps_results_TD.json").open(encoding="utf-8") as f:
    data              = json.load(f)
    low_correlations  = data["low_correlations"]
    high_correlations = data["high_correlations"]

# if you want numpy arrays back:
for d in low_correlations + high_correlations:
    for k in ("alpha0", "D0", "D1"):               # keys that were arrays
        d[k] = np.array(d[k])

In [ ]:
# ──────────────────────────────────────────────────────────────
#  Histograms for  E(T_D | T1 > t)
# ──────────────────────────────────────────────────────────────
# ------------------------------------------------------------
BIN_WIDTH        = 0.05     # fixed bin width       [k,k+1)
OUTLIER_FRACTION = 1e-8     # “rare”  ⇔  < tallest/20 observations
# ------------------------------------------------------------
def _analyse(vals, dict_list, key_name, title):
    """
    Build the histogram, classify bins, print rare-bin observations
    (sorted by descending value) and return the values that remain
    after outlier removal, ready to be plotted.
    """
    if not vals:                           # nothing to plot
        print(f"[{title}]  no observations.")
        return np.array([])

    vals = np.asarray(vals)
    # unit-width bin edges that cover the data
    lo, hi = np.floor(vals.min()), np.ceil(vals.max())
    edges  = np.arange(lo, hi + BIN_WIDTH + 1e-9, BIN_WIDTH)
    hist, _ = np.histogram(vals, bins=edges)

    tallest  = hist.max()
    rare_bins = np.where(hist < tallest * OUTLIER_FRACTION)[0]

    if rare_bins.size:                     # report every entry in a rare bin
        print(f"\n[{title}]  observations in sparse bins "
              f"(< {tallest/OUTLIER_FRACTION:.0f} items per bin):")

        bin_id   = np.searchsorted(edges, vals, side="right") - 1
        out_mask = np.isin(bin_id, rare_bins)
        out_idx  = np.where(out_mask)[0]

        # sort indices by *descending* value of the statistic
        out_idx  = out_idx[np.argsort(vals[out_idx])[::-1]]

        for i in out_idx:
            prm = dict_list[i]
            print(f"  • {key_name} = {vals[i]:.5g}")
            print(f"    alpha0 = {prm['alpha0']}")
            print(f"    D0 = {_pretty_matrix(prm['D0'])}")
            print(f"    D1 = {_pretty_matrix(prm['D1'])}")
            if "Correlations" in prm:
                print(f"    corr = {np.array(prm['Correlations'])}\n")

    return vals

def _stats_text(data):
    """Return a 3-line string with μ, median, c_v, skew."""
    μ  = data.mean()
    med = np.median(data)
    σ  = data.std(ddof=0)
    cv = σ/μ if μ else np.nan
    g  = 3*(μ-med)/σ if σ else np.nan
    return (f"mean                 = {μ:.3g}\n"
            f"median               = {med:.3g}\n"
            f"standard deviation   = {σ:.3g}\n"
            f"variation coeff      = {cv:.3g}\n"
            f"skewness             = {g:.3g}")

def _plot_unit_hist(ax, data, colour, label):
    ax.hist(
        data,
        bins=np.arange(np.floor(data.min()),
                       np.ceil(data.max()) + BIN_WIDTH + 1e-9,
                       BIN_WIDTH),
        edgecolor="k",
        color=colour,
        alpha=0.7,
        density=True,            # ← show *density* instead of frequency
    )
    ax.set_xlabel(label)
    ax.set_ylabel("density")     # ← y-axis label matches the new scale

    # ---- add statistics box ----
    ax.text(0.98, 0.98, _stats_text(data),
            transform=ax.transAxes, va="top", ha="right",
            bbox=dict(boxstyle="round,pad=0.3", fc="w", ec=colour, lw=1))

def plot_TD_conditional(low_dicts, high_dicts):
    """
    Two 1-unit-bin histograms of      μ_D = E(T_D | T1>t)

        left  : low-correlation models
        right : high-correlation models
    """

    # ----------- LOW-corr group -----------
    low_TD = _analyse([d["mean"] for d in low_dicts if "TD" in d],
                      low_dicts, "TD", "LOW  E(T_D | T1>t)")

    # ----------- HIGH-corr group ----------
    high_TD = _analyse([d["mean"] for d in high_dicts if "TD" in d],
                       high_dicts, "TD", "HIGH E(T_D | T1>t)")

    # ---------- plotting ----------
    if not low_TD.size and not high_TD.size:
        print("No TD data available to plot.");  return

    fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharey=True)

    if low_TD.size:
        _plot_unit_hist(ax[0], low_TD,  "gray", r"$\mathbb{E}[T_D \mid T_1>t]$")
        ax[0].set_title(f"Low correlations  (n = {len(low_TD)})")
    else:
        ax[0].set_visible(False)

    if high_TD.size:
        _plot_unit_hist(ax[1], high_TD, "steelblue",
                        r"$\mathbb{E}[T_D \mid T_1>t]$")
        ax[1].set_title(f"High correlations (n = {len(high_TD)})")
    else:
        ax[1].set_visible(False)

    plt.tight_layout()
    plt.show()

# Call it
plot_TD_conditional(low_correlations, high_correlations)

# Simulating MAPs

In [ ]:
RANDOM_SEED = 123456

def simulate_map(params, tmax, N_sim=1000, seed=None, show_all=False):
    """
    Simulates paths of a standard MAP (Markovian Arrival Process).

    Parameters:
    - params: dict with keys 'alpha0', 'D0', 'D1'
    - tmax: float, maximum simulation time per path
    - N_sim: int, number of simulations
    - seed: int or None, for reproducibility
    - show_all: bool, if True include times of all transitions (silent and arrivals)

    Returns:
    - List of dictionaries per simulation. Each dictionary contains:
        - 'arrivals': list of arrival times (T1, T2, ...)
        - 'all_transitions' (optional): list of (time, type) pairs (type = 'arrival' or 'silent')
    """
    # Set the random seed, if provided
    if seed is not None:
        np.random.seed(seed)

    # Extract MAP parameters from the dictionary
    alpha0 = params["alpha0"]
    D0 = params["D0"]
    D1 = params["D1"]
    m = len(alpha0)
    
    results = []
    for _ in range(N_sim):
        # Initial state
        current_state = np.random.choice(m, p=alpha0)
        
        events = []
        time = 0.0
        while time < tmax:
            # Calculate the rate
            rate = -D0[current_state, current_state]
            
            # Sample holding time
            holding_time = np.random.exponential(1 / rate)
            time += holding_time
            if time > tmax:
                break
            
            # Construct transition probabilities
            D0_row = D0[current_state, :].copy()
            D1_row = D1[current_state, :].copy()
            D0_row[current_state] = 0  # remove diagonal
            
            # Determine next transition (either silent or arrival)
            probs = -np.concatenate([D0_row, D1_row]) / rate
            probs /= probs.sum()  # renormalize

            # Sample next transition
            next_state = np.random.choice(2 * m, p=probs)
            
            if next_state < m:
                event_type = 'silent'
            else:
                next_state -= m
                event_type = 'arrival'

            # Record results
            if show_all or event_type == 'arrival':
                events.append({
                    'time': time,
                    'type': event_type,
                    'from_state': current_state,
                    'to_state': next_state
                })

            # Prepare the current state of the next iteration
            current_state = next_state

        # Obtain the final dataframe for the current run (run/iteration/patient)
        results.append(pd.DataFrame(events))

    # Flatten list if only 1 run is asked for
    if N_sim == 1:
        results = results[0]
        
    return results

In [ ]:
# Define MAP
alpha0 = np.array([0.5, 0.5])
D0 = np.array([[-2.5, 2], 
               [3, -10]])
D1 = np.array([[0, 0.5], 
               [7, 0]])
params = {"alpha0": alpha0, "D0": D0, "D1": D1}

# Simulate
results = simulate_map(params, tmax=3, N_sim=1, seed=12345, show_all=True)
results

In [ ]:
# Simulate one MAP path with full info
transitions_df = simulate_map(params, tmax=3.0, N_sim=1, seed=12345, show_all=True)

# Plotting the 3 horizontal timelines
fig, axs = plt.subplots(3, 1, figsize=(12, 6), sharex=True)
line_y = [3, 2, 1]  # Y positions
colors = {"arrival": "red", "silent": "blue"}

# Formatting tweaks
tick_height = 0.3
line_width = 4.0
label_fontsize = 10

# Helper to avoid overlapping by offsetting vertically
def vertical_offsets(times, base, spacing=0.15):
    seen = {}
    offsets = []
    for t in times:
        level = seen.get(t, 0)
        seen[t] = level + 1
        offsets.append(base + spacing * (level % 3))  # Cycle every 3 levels
    return offsets

# --- Line 1: Full Information (Silent + Arrival) ---
axs[0].hlines(line_y[0], 0, 3, colors='black', linewidth=line_width)
times1 = transitions_df["time"].tolist()
label_y1 = vertical_offsets(times1, line_y[0] + 0.25)
time_y1 = vertical_offsets(times1, line_y[0] - 0.5)

for (i, row), ly, ty in zip(transitions_df.iterrows(), label_y1, time_y1):
    axs[0].vlines(row["time"], line_y[0] - tick_height, line_y[0] + tick_height,
                  colors=colors[row["type"]], linewidth=line_width)
    axs[0].text(row["time"], ly, f"{row['to_state']}", ha="center", fontsize=label_fontsize)
    axs[0].text(row["time"], ty, f"{row['time']:.2f}", ha="center", fontsize=label_fontsize)

axs[0].vlines([0, 3], line_y[0] - tick_height, line_y[0] + tick_height, colors='black', linewidth=line_width)
axs[0].text(0, line_y[0] + 0.25, f"{transitions_df.iloc[0]['from_state']}", ha="center", fontsize=label_fontsize)
axs[0].set_yticks([line_y[0]])
axs[0].set_yticklabels(["Full Information"])

# --- Line 2: Observer + Chain ---
obs = transitions_df[transitions_df["type"] == "arrival"]
axs[1].hlines(line_y[1], 0, 3, colors='black', linewidth=line_width)
times2 = obs["time"].tolist()
label_y2 = vertical_offsets(times2, line_y[1] + 0.25)
time_y2 = vertical_offsets(times2, line_y[1] - 0.5)

for (i, row), ly, ty in zip(obs.iterrows(), label_y2, time_y2):
    axs[1].vlines(row["time"], line_y[1] - tick_height, line_y[1] + tick_height,
                  colors='red', linewidth=line_width)
    axs[1].text(row["time"], ly, f"{row['to_state']}", ha="center", fontsize=label_fontsize)
    axs[1].text(row["time"], ty, f"{row['time']:.2f}", ha="center", fontsize=label_fontsize)

axs[1].vlines([0, 3], line_y[1] - tick_height, line_y[1] + tick_height, colors='black', linewidth=line_width)
axs[1].text(0, line_y[1] + 0.25, f"{transitions_df.iloc[0]['from_state']}", ha="center", fontsize=label_fontsize)
axs[1].set_yticks([line_y[1]])
axs[1].set_yticklabels(["Observer + Chain"])

# --- Line 3: Arrivals Only ---
axs[2].hlines(line_y[2], 0, 3, colors='black', linewidth=line_width)
for i, row in obs.iterrows():
    axs[2].vlines(row["time"], line_y[2] - tick_height, line_y[2] + tick_height,
                  colors='red', linewidth=line_width)
    axs[2].text(row["time"], line_y[2] - 0.5, f"{row['time']:.2f}", ha="center", fontsize=label_fontsize)

axs[2].vlines([0, 3], line_y[2] - tick_height, line_y[2] + tick_height, colors='black', linewidth=line_width)
axs[2].set_yticks([line_y[2]])
axs[2].set_yticklabels(["Arrivals Only"])

# --- Final Layout ---
for ax in axs:
    ax.set_xlim(0, 3)
    ax.set_ylim(0.25, 3.75)
    ax.set_xticks(np.arange(0, 3.1, 0.5))
    ax.grid(True, axis='x', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()